In [1]:
import pandas as pd
from tqdm import tqdm
import time
import numpy as np

from __future__ import annotations
from yandex_cloud_ml_sdk import YCloudML


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("display.max_colwidth", None)

In [2]:
X_test = pd.read_csv('X_test_balanced.csv',index_col=0)

In [3]:
X_test.head(10)

,purpose
393593,Перечисление средств по соглашению №2140 о предоставлении гранта победителю конкурса на предоставление грантов Раиса РТ на развитие гражданского общества от 26.11.2024г. НДС не облагается.
429170,ЗА 27/02/2025;СУМ:1500.00;КОМ:0.00;СЛЮСАРЕНКО ВИКТОРИЯ АНАТОЛЬЕВНА;Попечительский взнос котёнок Тужур
379020,"Перевод с карты *6947, Благотворительное пожертвование на уставную деятельность. НДС не облагается"
570841,"ОПЛАТА ПО ДОГОВОРУ НХТК.8190 (ДС НХТК.8190-2), СЧЕТ №21 ОТ 02.06.2025 (КВОТИРУЕМЫЕ РАБОЧИЕ МЕСТА). СУММА 80143-68 РУБ. БЕЗ НАЛОГА (НДС)"
485026,БЛАГОТВОРИТЕЛЬНАЯ ПОМОЩЬ ПО СОГЛАШЕНИЮ №БС2025-005 ОТ 16.01.2025 СУММА 310750-00 БЕЗ НАЛОГА (НДС)
591200,Выплата процентов по депозитной сделке № UNV-0000006342917 от 22.08.2025 за период с 23.08.2025 по 25.08.2025 согласно Правилам Банковского обслуживания. НДС не предусмотрен.
396289,"Перевод с карты *3120, Благотворительное пожертвование на уставную деятельность. НДС не облагается"
402359,Благотворительное пожертвование на уставную деятельность. НДС не облагается
147304,"Оплата по договору о предоставлении целевого гранта в рамках Благотворительной программы - акции ""Добрый новогодний подарок"" (2023-2024) № 24-1-000496 от 15.04.2024. 1 транш. НДС не облагается."
171375,Благотворительное пожертвование по договору 111-77/2024 от 29.02.2024НДС не облагается


In [4]:
y_pred_ygpt = pd.DataFrame(columns=["universal_category"])

sdk = YCloudML(
        folder_id="YOUR_FOLDER_ID",
        auth="YOUR_YANDEX_CLOUD_TOKEN",
    )

task_description="""Представь, что ты обученная модель для классификации благотворитрельных платежей по нескольким универсальным категориям доходных статей.
                    Есть датасет с тренировочными данными - X_train.csv и y_train.csv – с целевым признаком для классификации по нескольким категориям, изучи внимательно тренировочные данные и далее я буду давать тебе тестовые строки - нужно предложить одну из искомых категорий статей клаccификации из общего набора:
                    пожертвования от физических лиц (напрямую)
                    пожертвования через платформы
                    продажа товаров
                    прочие недоходные операции
                    продажа услуг
                    пожертвования от юридических лиц (напрямую)
                    финансовые доходы
                    членские и учредительские взносы
                    гранты субсидии конкурсы
                    Обрати внимание, что в тренировочных данных миксплат, юмани относятся к категориям не платформ а частных пожертвований, потому что это технические способы сбора средств частных пожертвований. Платформы - это специальные благотворительные программы, сайты, заточенные именно сбор целевых или благотворительных платежей, а миксплат, юмани, различные сайты с наличием слова "деньги" в названиях - это обычные платежные системы.
                    Важно, например тбанк и тбанк благотворительность - это очень разные сферы, первое - это просто финансовый канал и сам по себе он относится к поступлениям от физ. лиц, а второе в это именно благотворительная платформа тбанка – смотри внимательно и соотноси названия статей и универсальные категории в тренировочной выборке.
                    Если будет необходимо - сходи в поиск и проверь названия в интернете
                    Как будешь готов предложи универсальные категории для платежей из X_test.csv и верни назад список в формате csv с двумя колонками - id соответствующий платежу в X_test и твой прогноз по категории. Не пропусти ни одного платежа - точно каждому назначь универсальную категорию, приложи все усилия, и будь очень внимателен к редким классам - данные по ним постарались выложить максимально полные в тренировочной выборке.
                    действуй строго по списку выше, не придумывай дополнительных категорий и дай ответы ко всем платежам из X_test.csv - строго по их id, проверяя не дублируются ли id и не перепутай входные тексты""".replace("\n", " ").strip()
labels = [
    "пожертвования от физических лиц (напрямую)",
    "пожертвования через платформы",
    "продажа товаров",
    "прочие недоходные операции",
    "продажа услуг",
    "пожертвования от юридических лиц (напрямую)",
    "финансовые доходы",
    "членские и учредительские взносы",
    "гранты субсидии конкурсы"
]

import numpy as np
import time
from tqdm import tqdm

# Создаём модель один раз
model = sdk.models.text_classifiers("yandexgpt").configure(
    task_description=task_description,
    labels=labels,
)

for index_, text in tqdm(X_test['purpose'].items(), total=len(X_test)):
    attempts = 0
    max_attempts = 3
    while attempts < max_attempts:
        try:
            result = model.run(str(text))  # приводим текст к строке
            best_label = max(result, key=lambda x: x.confidence).label
            y_pred_ygpt.loc[index_, 'universal_category'] = best_label
            break
        except Exception as e:
            attempts += 1
            print(f"Ошибка на index {index_}, попытка {attempts}/{max_attempts}: {e}")
            time.sleep(2)
    else:
        # если все попытки неудачные, присваиваем NaN
        y_pred_ygpt.loc[index_, 'universal_category'] = np.nan

y_pred_ygpt.to_csv("y_pred_ygpt.csv", index=True, header=["universal_category"])
    

100%|██████████| 900/900 [30:58<00:00,  2.07s/it]


In [5]:
y_pred_ygpt.to_csv("y_pred_ygpt.csv", index=True, header=["universal_category"])